### `phis_contour.ipynb` 
*Created: September 22, 2026* <br/>
This notebook implements the contour integral versions of the first four phi functions. Recall that the $m$-th Phi function $(m \geq 0)$ is 
\begin{align*}
   \varphi_m(z) &= \sum_{k=0}^{\infty} \frac{z^k}{(k+m)!} = z^{-m}\left(e^z - 1 - z - \frac{z^2}{2!} - \cdots - \frac{z^{m-1}}{(m-1)!} \right)   \qquad \textrm{when } z \neq 0
\end{align*}

We define $\varphi_m(0) := 0$. The implementation in this notebook evaluates $\varphi_m(z)$ using the Taylor series when $|z|$ is small. If $|z|$ is not small, then $\varphi_m(z)$ is evaluated using the recursive formula

$$\phi_{m+1}(z) = \frac{\phi_m(z) - \frac{1}{m!}}{z}$$

In [1]:
using LinearAlgebra, NBInclude, LaTeXStrings, CairoMakie, UnPack
@nbinclude("../../../../set_makie_defaults.ipynb")

In [2]:
"""The Phi Functions (directly from the definitions)

    NOTES
    -----
     -  These implementations should NOT be considered reliable in their own right.
        They are intended only for the contour integration formulas implemented later.   
     -  The four phi functions below are valid for Float64 or ComplexF64 scalar inputs. 
     -  The phi functions satisfy ϕₖ(0) = 1/k! for each k ≥ 0. 
"""

#Naive implementations
ϕ₀(z) = exp(z)
ϕ₁(z) = iszero(z) ? one(z)   : expm1(z) / z 
ϕ₂(z) = iszero(z) ? one(z)/2 : (expm1(z) - z) / z^2 
ϕ₃(z) = iszero(z) ? one(z)/6 : (expm1(z) - z - z^2/2) / z^3

ϕ₃ (generic function with 1 method)

In [3]:
#COMPLEX SCALAR version
function phi_contour(λ::ComplexF64, k::Int; M::Int = 64)
    """
    Evaluate φₖ at the complex number `λ` using contour integration: 
    
        φₖ(λ) = 1/(2*π*i) * ∫_{Γ} ϕₖ(z) / (z-λ) dz 

    where Γ is the circle in the complex plane with radius 1, centered at λ = x₀ + iy₀. 
    Γ is parameterized by γ(θ) = λ + cos(θ) + i sin(θ) = x₀ + cos(θ) + i (y₀ + sin(θ)), 0 ≤ θ ≤ 2π. 
    We end up with the formula 

                    φₖ(λ) = 1/(2π) * ∫_{0}^{2π} ϕₖ(λ + e^{iθ}) dθ
       
    PARAMETERS
    ----------
    λ :: complex number at which to evaluate the phi function
    k :: the phi function subscript
    M :: number of quadrature nodes 
    """

    M > 0 || throw(ArgumentError("M must be positive."))
    0 ≤ k ≤ 3 || throw(ArgumentError("k must be 0,1,2, or 3."))
    
    #Quadrature points (equally spaced angles on [0,2π], offset by Δθ/2 = π/M)
    θ = (2π/M) .* (1/2:(M-1/2))

    #radius of contour circle 
    r = 1.0 

    #Contour points
    z = λ .+ r .* exp.(im .* θ)
  
    if k == 0 
        return sum(ϕ₀.(z)) / M
    elseif k == 1 
        return sum(ϕ₁.(z)) / M
    elseif k == 2 
        return sum(ϕ₂.(z)) / M
    else 
        return sum(ϕ₃.(z)) / M
    end 
end

phi_contour (generic function with 1 method)

In [4]:
#REAL SCALAR version
function phi_contour(λ::Float64, k::Int; M::Int = 64)
    """
    TODO: Test accuracy of phi_contour by comparing against a high precision reference implementation. 
    If the relative error is very small, then that means `phi_contour` is trustworthy. 
    
    Compute φₖ(λ) using contour integration. The sample points are evaluated using the direct formulas given above.
    
    PARAMETERS
    ----------
    λ :: real number at which to evaluate the phi function
    k :: the phi function subscript
    M :: number of quadrature nodes 
    """

    #Note: For a real argument lambda, we can leverage symmetry and just integrate over the upper semi-circle. 
    #"If this symmetry is not explicitly used in the computation of the ϕ-functions when λ is real, rounding
    #errors appear that lead to numerical instability." - Montanelli & Bootland, 2016. 

    #**See my phi function / contour integration notes for details on the symmetry property we are using.**

    M > 0 || throw(ArgumentError("M must be positive."))
    0 ≤ k ≤ 3 || throw(ArgumentError("k must be 0,1,2 or 3."))
    
    #angles of the quadrature points 
    θ = (π/M) .* (0.5:1.0:(M-0.5))

    #radius of contour circle 
    r = 1.0 

    #Contour points
    z = λ .+ r .* exp.(im .* θ)
  
    if k == 0 
        return real(sum(ϕ₀.(z)) / M)
    elseif k == 1 
        return real(sum(ϕ₁.(z)) / M)
    elseif k == 2 
        return real(sum(ϕ₂.(z)) / M)
    else
        return real(sum(ϕ₃.(z)) / M)
    end 
end

phi_contour (generic function with 2 methods)

In [5]:
#DIAGONAL MATRIX version
function phi_contour(B::Diagonal, k::Int; M::Int = 64)
    """
    Compute ϕₖ(B), where B is a digonal matrix 
    Supports either B::Diagonal{Float64} or B::Diagonal{ComplexF64}
    """
   
    M > 0 || throw(ArgumentError("M must be positive."))
    0 ≤ k ≤ 3 || throw(ArgumentError("k must be 0,1,2, or 3."))
    
    phis = phi_contour.(B.diag, k; M)
    return Diagonal(phis)
end

phi_contour (generic function with 3 methods)

In [6]:
#Contour versions of Phi functions 
φ₀(λ; M::Int = 64) = phi_contour(λ, 0; M = M)
φ₁(λ; M::Int = 64) = phi_contour(λ, 1; M = M)
φ₂(λ; M::Int = 64) = phi_contour(λ, 2; M = M)
φ₃(λ; M::Int = 64) = phi_contour(λ, 3; M = M)

φ₃ (generic function with 1 method)